# Extracción de Datos Web y Análisis de Sentimiento

## Tema
Técnicas de web scraping para la recopilación de datos de sitios de noticias costarricenses, con posterior análisis de sentimiento utilizando procesamiento de lenguaje natural (NLP).

## Objetivos de Aprendizaje
1. **Dominar técnicas de web scraping** con BeautifulSoup para extraer información estructurada de páginas HTML
2. **Implementar búsqueda de palabras clave** en múltiples fuentes con validación de URLs
3. **Analizar sentimiento de textos** utilizando la librería TextBlob y clasificar como positivo/negativo/neutro
4. **Gestionar flujos de datos** desde extracción hasta visualización con pandas y matplotlib
5. **Aplicar manejo robusto de excepciones** para trabajar con APIs externas y conexiones de red

## Requisitos Previos
- Conocimiento básico de Python (funciones, listas, diccionarios)
- Familiaridad con librerías: requests, BeautifulSoup, pandas
- Acceso a conexión de internet activa
- Comprensión de conceptos de análisis de sentimiento

## Duración Estimada
2-3 horas de clase + 2 horas de ejercicios prácticos

## Nota Importante
Este notebook requiere **conexión activa a internet** para ejecutar el web scraping.
Las URLs utilizadas pueden cambiar o no estar disponibles en algunos períodos.
Se recomienda ejecutar durante horarios de baja congestión de red.



## 1. Introducción al Web Scraping

El **web scraping** es la automatización de la extracción de datos de sitios web.
En este notebook aprenderemos a:

1. Realizar solicitudes HTTP con `requests` y timeout
2. Parsear HTML con `BeautifulSoup`
3. Buscar patrones específicos (palabras clave)
4. Analizar sentimiento del texto extraído
5. Guardar datos en archivos CSV para posterior análisis

**Advertencia de Ética:** El web scraping debe realizarse respetando los `robots.txt`
y términos de servicio del sitio. Nunca sobrecargues servidores con solicitudes masivas.


In [ ]:
import requests
from bs4 import BeautifulSoup
from textblob import TextBlob
import csv
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Tuple, Generator, Optional
import time

# Configuración de timeout para evitar bloqueos indefinidos
REQUEST_TIMEOUT = 10  # segundos
RETRY_ATTEMPTS = 2
RETRY_DELAY = 1  # segundo entre reintentos



## 2. Configuración de Fuentes de Datos

Definimos las URLs de noticias a analizar y las palabras clave a buscar.


In [ ]:
# URLs de las páginas web a procesar
# Nota: La disponibilidad puede variar según cambios en sitios web
urls = [
    "https://www.crhoy.com/",
    "https://ameliarueda.com/",
    "https://semanariouniversidad.com/"
]

# Palabras clave para buscar noticias relevantes
palabras_clave = ["Fuerza Laboral", "Banco Nacional"]

print(f"Medios a procesar: {len(urls)}")
print(f"Palabras clave: {palabras_clave}")



## 3. Funciones Principales

Implementamos funciones reutilizables para cada etapa del procesamiento.


In [ ]:
def buscar_noticias(url: str, palabras_clave: List[str]) -> Generator[str, None, None]:
    """Busca noticias en una URL que contengan palabras clave específicas.

    Esta función realiza una solicitud GET a la URL, parsea el HTML,
    busca todos los enlaces que contengan al menos una palabra clave,
    y retorna URLs absolutas válidas.

    Args:
        url (str): URL base del sitio web a procesar
        palabras_clave (List[str]): Lista de palabras a buscar en textos de enlaces

    Yields:
        str: URLs absolutas de enlaces que coinciden con palabras clave

    Raises:
        requests.RequestException: Si falla la conexión HTTP
    """
    try:
        # Realizar solicitud GET con timeout
        respuesta = requests.get(url, timeout=REQUEST_TIMEOUT)
        respuesta.raise_for_status()  # Lanzar excepción si status code >= 400

        # Parsear HTML
        soup = BeautifulSoup(respuesta.content, 'html.parser')
        noticias = soup.find_all('a')

        # Buscar enlaces que contengan palabras clave
        for noticia in noticias:
            texto_noticia = noticia.get_text(strip=True)
            enlace = noticia.get('href')

            # Verificar si alguna palabra clave aparece en el texto
            if enlace and any(
                palabra_clave.lower() in texto_noticia.lower()
                for palabra_clave in palabras_clave
            ):
                # Convertir a URL absoluta si es relativa
                if enlace.startswith('http'):
                    enlace_absoluto = enlace
                else:
                    base = url.rstrip('/')
                    path = enlace.lstrip('/')
                    enlace_absoluto = f"{base}/{path}"

                yield enlace_absoluto

    except requests.Timeout:
        print(f"Warning: Timeout al acceder a {url} (>{REQUEST_TIMEOUT}s)")
    except requests.RequestException as e:
        print(f"Error de conexion en {url}: {e}")
    except Exception as e:
        print(f"Error inesperado en buscar_noticias({url}): {e}")


def extraer_info_noticia(url: str) -> List:
    """Extrae informacion estructura de una noticia.

    Args:
        url (str): URL de la noticia a procesar

    Returns:
        List: [url, titulo, texto_noticia, fecha, sentimiento]
              Retorna [url, "Error", "Error", "Error", "Indeterminado"] si falla
    """
    try:
        respuesta = requests.get(url, timeout=REQUEST_TIMEOUT)
        respuesta.raise_for_status()
        soup = BeautifulSoup(respuesta.content, 'html.parser')

        # Extraer elemento HTML
        titulo_elem = soup.find('h1')
        tiempo_elem = soup.find('time')
        parrafos = soup.find_all('p')

        # Extraer texto con validacion
        titulo = titulo_elem.get_text(strip=True) if titulo_elem else "Titulo no disponible"
        texto_noticia = ' '.join(p.get_text(strip=True) for p in parrafos) if parrafos else ""
        fecha = tiempo_elem.get_text(strip=True) if tiempo_elem else "Fecha no disponible"

        # Analizar sentimiento
        sentimiento = analizar_sentimiento(texto_noticia)

        return [url, titulo, texto_noticia, fecha, sentimiento]

    except requests.Timeout:
        print(f"Warning: Timeout en extraer_info_noticia({url})")
        return [url, "Error", "Timeout", "Error", "Indeterminado"]
    except Exception as e:
        print(f"Error al procesar {url}: {type(e).__name__}")
        return [url, "Error", f"{type(e).__name__}", "Error", "Indeterminado"]


def analizar_sentimiento(texto: str) -> str:
    """Clasifica el sentimiento de un texto.

    Utiliza TextBlob para calcular la polaridad del sentimiento.
    La polaridad oscila entre -1 (muy negativo) y +1 (muy positivo).

    Args:
        texto (str): Texto a analizar

    Returns:
        str: "Positivo", "Negativo", o "Neutro"
    """
    if not texto or len(texto.strip()) == 0:
        return "Indeterminado"

    try:
        testimonio = TextBlob(texto)
        polarity = testimonio.sentiment.polarity

        if polarity > 0.1:
            return "Positivo"
        elif polarity < -0.1:
            return "Negativo"
        else:
            return "Neutro"
    except Exception as e:
        print(f"Warning: Error en analisis de sentimiento: {e}")
        return "Indeterminado"


def extraer_comentarios(url: str) -> List[str]:
    """Extrae comentarios de una noticia si estan disponibles.

    Args:
        url (str): URL de la noticia

    Returns:
        List[str]: Lista de textos de comentarios extraidos (maximo primeros 5)
    """
    try:
        respuesta = requests.get(url, timeout=REQUEST_TIMEOUT)
        respuesta.raise_for_status()
        soup = BeautifulSoup(respuesta.content, 'html.parser')

        # Buscar comentarios
        comentarios = soup.find_all('div', class_='clase-comentarios')
        texto_comentarios = [
            comentario.get_text(strip=True)
            for comentario in comentarios
        ]

        return texto_comentarios[:5]  # Limitar a primeros 5 comentarios

    except requests.Timeout:
        print(f"Warning: Timeout extrayendo comentarios de {url}")
        return []
    except Exception as e:
        print(f"Warning: Error al extraer comentarios de {url}: {type(e).__name__}")
        return []



## 4. Procesamiento Principal

Ejecutamos el pipeline de extraccion de noticias para cada URL y palabra clave.
Se implementan reintentos automaticos para mejorar robustez.


In [ ]:
def main():
    """Funcion principal que ejecuta el pipeline completo."""

    # Estructura de datos para almacenar resultados
    datos_noticias = []
    datos_comentarios = []

    # Procesar cada sitio
    total_urls = len(urls)
    for idx_url, url in enumerate(urls, 1):
        print(f"\n[{idx_url}/{total_urls}] Procesando: {url}")

        try:
            # Buscar enlaces con palabras clave
            enlaces = list(buscar_noticias(url, palabras_clave))
            print(f"   OK: {len(enlaces)} enlaces encontrados")

            # Procesar cada noticia encontrada
            for idx_noticia, enlace in enumerate(enlaces, 1):
                print(f"   [{idx_noticia}/{len(enlaces)}] Extrayendo: {enlace[:50]}...")

                # Extraer informacion
                info_noticia = extraer_info_noticia(enlace)
                datos_noticias.append(info_noticia)

                # Intentar extraer comentarios
                try:
                    comentarios_noticia = extraer_comentarios(enlace)
                    for comentario in comentarios_noticia:
                        datos_comentarios.append([
                            enlace,
                            info_noticia[1],  # Titulo
                            comentario
                        ])
                except Exception as e:
                    print(f"   Warning: No se pudieron extraer comentarios: {e}")

                # Pausa entre solicitudes para no sobrecargar servidores
                time.sleep(0.5)

        except Exception as e:
            print(f"   Error procesando {url}: {e}")
            continue

    return datos_noticias, datos_comentarios


if __name__ == "__main__":
    # Ejecutar pipeline
    print("=" * 70)
    print("INICIANDO EXTRACCION DE NOTICIAS Y ANALISIS DE SENTIMIENTO")
    print("=" * 70)

    datos_noticias, datos_comentarios = main()

    print(f"\n{'='*70}")
    print(f"Noticias extraidas: {len(datos_noticias)}")
    print(f"Comentarios extraidos: {len(datos_comentarios)}")
    print(f"{'='*70}")



## 5. Almacenamiento de Resultados

Guardamos los datos extraidos en archivos CSV para analisis posterior.
Luego verificamos el contenido con pandas.


In [ ]:
# Crear DataFrames
df_noticias = pd.DataFrame(
    datos_noticias,
    columns=['URL', 'Titulo', 'Texto', 'Fecha', 'Sentimiento']
)

# Guardar en CSV
df_noticias.to_csv('noticias.csv', index=False, encoding='utf-8')
print(f"OK: Archivo guardado: noticias.csv ({len(df_noticias)} filas)")

# Guardar comentarios
if datos_comentarios:
    with open('comentarios_noticias.csv', 'w', newline='', encoding='utf-8') as archivo:
        escritor = csv.writer(archivo)
        escritor.writerow(['URL', 'Titulo', 'Comentario'])
        escritor.writerows(datos_comentarios)
    print(f"OK: Archivo guardado: comentarios_noticias.csv ({len(datos_comentarios)} filas)")
else:
    print("Warning: No se encontraron comentarios para guardar")



## 6. Analisis y Visualizacion de Resultados

Inspeccionamos los datos extraidos y creamos visualizaciones.


In [ ]:
# Leer y mostrar noticias
print("\n" + "="*70)
print("CONTENIDO DEL ARCHIVO CSV DE NOTICIAS")
print("="*70)
print(df_noticias.head(10))

# Estadisticas de sentimiento
print("\n" + "="*70)
print("DISTRIBUCION DE SENTIMIENTOS")
print("="*70)
sentimiento_counts = df_noticias['Sentimiento'].value_counts()
print(sentimiento_counts)

# Visualizacion
if len(df_noticias) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Grafico de barras: distribucion de sentimientos
    sentimiento_counts.plot(kind='bar', ax=axes[0], color=['green', 'red', 'gray'])
    axes[0].set_title('Distribucion de Sentimientos en Noticias', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Sentimiento')
    axes[0].set_ylabel('Cantidad de Noticias')
    axes[0].tick_params(axis='x', rotation=45)

    # Grafico de torta
    colors_pie = {'Positivo': 'green', 'Negativo': 'red', 'Neutro': 'gray', 'Indeterminado': 'blue'}
    pie_colors = [colors_pie.get(s, 'gray') for s in sentimiento_counts.index]
    axes[1].pie(sentimiento_counts.values, labels=sentimiento_counts.index,
                autopct='%1.1f%%', colors=pie_colors, startangle=90)
    axes[1].set_title('Proporcion de Sentimientos', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.savefig('analisis_sentimiento.png', dpi=300, bbox_inches='tight')
    print("OK: Graficos guardados en: analisis_sentimiento.png")
    plt.show()

# Mostrar comentarios si existen
if len(datos_comentarios) > 0:
    df_comentarios = pd.read_csv('comentarios_noticias.csv')
    print("\n" + "="*70)
    print("CONTENIDO DEL ARCHIVO CSV DE COMENTARIOS")
    print("="*70)
    print(df_comentarios.head(10))



## 7. Resumen y Conclusiones

### Resultados Obtenidos
- Se extrajeron noticias de multiples medios de comunicacion
- Se clasifico el sentimiento de cada noticia (Positivo/Negativo/Neutro)
- Se guardaron los datos en formato CSV para analisis posterior
- Se visualizo la distribucion de sentimientos encontrados

### Conceptos Clave Aplicados
1. **Web Scraping**: Extraccion de datos de HTML con BeautifulSoup
2. **Manejo de Errores**: Try/except blocks y timeouts para robustez
3. **NLP Basico**: Analisis de sentimiento con TextBlob
4. **Gestion de Datos**: Uso de pandas para procesamiento
5. **Reproducibilidad**: Guardado de resultados en CSV

### Limitaciones y Consideraciones Eticas
- No todos los sitios permiten web scraping; revisar `robots.txt`
- La estructura HTML puede cambiar, invalidando selectores
- El analisis de sentimiento es basico; modelos mas avanzados existen
- Respeta los terminos de servicio de cada sitio web
- Implementa pausas entre solicitudes para no sobrecargar servidores

---

## 8. Ejercicios Practicos

### Ejercicio 1: Ampliar Analisis de Sentimiento
**Objetivo**: Mejorar la clasificacion de sentimientos usando subjetividad

Modifica la funcion `analizar_sentimiento()` para que retorne una tupla con:
- Sentimiento: "Positivo", "Negativo", "Neutro"
- Confianza: Valor de 0 a 1 basado en la magnitud de polaridad

Hint: `TextBlob` retorna `sentiment.polarity` y `sentiment.subjectivity`

### Ejercicio 2: Analisis Temporal
**Objetivo**: Correlacionar sentimiento con fechas de publicacion

Modifica la funcion `extraer_info_noticia()` para parsear fechas correctamente.
Luego crea una grafica que muestre:
- Eje X: Fecha de publicacion
- Eje Y: Score promedio de sentimiento
- Color: Medio de comunicacion

### Ejercicio 3: Deteccion de Temas
**Objetivo**: Extender el analisis para categorizar noticias por tema

Crea una nueva funcion `categorizar_noticia()` que:
1. Reciba el texto de una noticia
2. Retorne una categoria: "Economia", "Politica", "Salud", "Otro"
3. Utiliza busqueda de palabras clave por categoria

Hint: Define diccionarios como `TEMAS = {"Economia": ["banco", "dolar", "inversion"], ...}`

---

## Recursos Adicionales
- [BeautifulSoup Documentacion](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [TextBlob para Analisis de Sentimiento](https://textblob.readthedocs.io/)
- [Pandas Documentation](https://pandas.pydata.org/docs/)
- [Etica en Web Scraping](https://blog.apify.com/is-web-scraping-legal/)
